Place for examples of capabilities. The following cell is the most reliable way I've found to accomplish the desired behavior, but it might be overkill.

In [1]:
using Pkg
Pkg.activate("..")
Pkg.resolve()
Pkg.instantiate()
Pkg.precompile()


  Activating project at `c:\Users\Will\.julia\dev\HybridDynamics.jl`
     Project No packages added to or removed from `C:\Users\Will\.julia\dev\HybridDynamics.jl\Project.toml`
    Manifest No packages added to or removed from `C:\Users\Will\.julia\dev\HybridDynamics.jl\Manifest.toml`
Precompiling packages...
   2254.0 ms  ✓ HybridDynamics
  1 dependency successfully precompiled in 2 seconds. 21 already precompiled.


In [2]:
import HybridDynamics as HD

In [3]:
import Plots as plt
using LaTeXStrings

In [ ]:
M(q) = [1.0 0.0;0.0 1.0]
V(q) = -q[2]
h(q) = 1 - (q[1]^2 + q[2]^2)
∇h(q) = [-2*q[1], -2*q[2]]
sysM = HD.MechanicalSystem(M, V; guard=h, normal = ∇h)

HybridDynamics.MechanicalSystem{typeof(M), typeof(V), typeof(h), typeof(∇h), HybridDynamics.var"#MechanicalSystem##2#MechanicalSystem##3", Float64}(Main.M, Main.V, Main.h, Main.∇h, HybridDynamics.var"#MechanicalSystem##2#MechanicalSystem##3"(), 1.0)

In [5]:
probM = HD.prob(sysM, [-0.6, 0.0, 0.0, 0.0], (0.0, 5.0))
solM = HD.solve(probM, solver=HD.RK4())

Float64


MethodError: MethodError: no method matching /(::Float64, ::LinearAlgebra.Adjoint{Float64, Matrix{Float64}})
The function `/` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  /(!Matched::LinearAlgebra.Transpose{T, <:AbstractVector} where T, ::LinearAlgebra.Adjoint{T, <:AbstractMatrix} where T)
   @ LinearAlgebra C:\Users\Will\.julia\juliaup\julia-1.12.6+0.x64.w64.mingw32\share\julia\stdlib\v1.12\LinearAlgebra\src\adjtrans.jl:529
  /(!Matched::LinearAlgebra.Transpose{T, <:AbstractVector} where T, ::AbstractMatrix)
   @ LinearAlgebra C:\Users\Will\.julia\juliaup\julia-1.12.6+0.x64.w64.mingw32\share\julia\stdlib\v1.12\LinearAlgebra\src\adjtrans.jl:527
  /(!Matched::LinearAlgebra.AdjointQ{<:Any, <:LinearAlgebra.LQPackedQ}, ::AbstractVecOrMat)
   @ LinearAlgebra C:\Users\Will\.julia\juliaup\julia-1.12.6+0.x64.w64.mingw32\share\julia\stdlib\v1.12\LinearAlgebra\src\abstractq.jl:620
  ...


In [ ]:
init = [1.,0.];
tspan = (0., 5.);

function M(q)
    [1.0;;]
end

function V(q)
    q[1]^2
end

# Table/guard
g(q) = q[1]

Lsys = HD.LagSys(M, V; guard = g, e=1.0)

probl = HD.prob(Lsys, init, tspan)
sl = HD.solve(probl, solver = HD.RK4())


In [ ]:
print(sl.t)

How tf am I going backwards in time...

In [ ]:
print(sl.jump_times)

In [ ]:
q_vals = getindex.(sl.x, 1)

pl2 = plt.plot(sl.t, q_vals,
    title = "Bouncing Ball",
    xlabel = "t",
    ylabel = "q",
    label = ""
)

display(pl2)

In [ ]:
# Attempt a Filippov system
F(x) = [3, -1]
G(x) = [0, 1]
H(x) = x[2]-sin(x[1])
N(x) = [-cos(x[1]), 1]

Fsys = HD.FilippovSys(F, G, H, N)
probF = HD.prob(Fsys, [0.0, 1.0], (0.0, 10.0))
solF = HD.solve(probF, HD.RK4(); dt_initial=0.01)

xf = getindex.(solF.x, 1);
yf = getindex.(solF.x, 2);
xh = range(minimum(xf)-0.5,
           maximum(xf)+0.5,
           length=1000);

In [ ]:
plt.plot(xf, yf, lw = 2, label = "Trajectory")
plt.plot!(xh, sin.(xh), lw = 1, lc =:black, ls =:dash, label = L"H(x) = 0")
plt.plot!(title = "Filippov Trajectory", xlabel = L"x", ylabel = L"y", dpi = 500)